# Build & Push Docker Image to ghcr.io
Run on Colab: Runtime → Run all

## Prerequisite
Create a GitHub **Personal Access Token (classic)** with scope `write:packages`:
1. https://github.com/settings/tokens → Generate new token (classic)
2. Select `write:packages`
3. Copy the token, paste when prompted below

In [ ]:
from getpass import getpass
import os, subprocess, sys

GH_USER = "effame"
GH_TOKEN = getpass("GitHub PAT (write:packages): ")
REPO = "effame/stockgen-upscaler"
IMAGE = f"ghcr.io/{REPO}:latest"

# Make vars available to shell commands
os.environ["GH_USER"] = GH_USER
os.environ["GH_TOKEN"] = GH_TOKEN
os.environ["REPO"] = REPO
os.environ["IMAGE"] = IMAGE
print("✓ Variables set")

In [ ]:
# Clone repo
os.chdir("/content")
!rm -rf stockgen-upscaler
!git clone https://github.com/{REPO}.git
%cd stockgen-upscaler
print("✓ Repo cloned")

In [ ]:
!git log --oneline -3
print("✓ Latest commits")

In [ ]:
# Install & start Docker using official script
!curl -fsSL https://get.docker.com -o /tmp/get-docker.sh && sh /tmp/get-docker.sh > /dev/null 2>&1
!dockerd > /dev/null 2>&1 &
import time; time.sleep(8)
!docker info > /dev/null 2>&1 && echo "✓ Docker ready" || (cat /var/log/docker.log 2>/dev/null; echo "✗ Docker failed")

In [ ]:
# Login to ghcr.io
!echo "$GH_TOKEN" | docker login ghcr.io -u "$GH_USER" --password-stdin && echo "✓ Logged in to ghcr.io"

In [ ]:
# Build Docker image (this takes a few minutes)
!docker build -t {IMAGE} . 2>&1
print("✓ Build complete")

In [ ]:
# Push to ghcr.io
!docker push {IMAGE} 2>&1 && echo "✓ Pushed"

In [ ]:
# Tag with commit SHA and push
SHA = !git rev-parse --short HEAD
sha = SHA[0]
!docker tag {IMAGE} ghcr.io/{REPO}:{sha}
!docker push ghcr.io/{REPO}:{sha} 2>&1
print(f"✓ Also pushed tag: {sha}")
print(f"\nImage: ghcr.io/{REPO}:latest")